# V27 — Rocket Landing, Colab T4 Training Run

**Target:** 98.5% – 99.3% precise landings under full domain randomization.

**Baseline to beat:** V23 @ 96.9% precise (current production).

**Design:** From-scratch SAC, [512,512] policy, 8-frame temporal stacking, 4 parallel envs, 8M steps, 3-phase curriculum. Reward is pure V23 — no modifications.

**Wall-clock estimate on Colab T4:** 10 – 12 h.

---

## Before you run

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Put your GitHub username in Cell 3 (`GITHUB_USER`). The repo must contain the `src/` folder with the training files.
3. Run Cell 1 → Cell 5 in order. Cell 5 is the 200k wiring test — **do NOT skip it**.
4. Only when the wiring test shows the curriculum printing and TensorBoard writing, run Cell 6 (the full 8M run).
5. Checkpoints land in `/content/drive/MyDrive/rocket/models/`. If Colab disconnects, you lose at most 100k steps.

## Cell 1 — Check GPU and mount Google Drive

In [ ]:
!nvidia-smi | head -20

from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/rocket/models /content/drive/MyDrive/rocket/logs
!ls /content/drive/MyDrive/rocket

## Cell 2 — Install pinned dependencies

Pinned to the exact versions V23 was trained with. Do not upgrade — it can silently change SAC behavior.

In [ ]:
!pip install -q \
    stable-baselines3[extra]==2.3.2 \
    sb3-contrib==2.3.0 \
    gymnasium[box2d]==1.0.0 \
    swig \
    tensorboard

import stable_baselines3, gymnasium, torch
print(f'SB3     : {stable_baselines3.__version__}')
print(f'Gym     : {gymnasium.__version__}')
print(f'Torch   : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Cell 3 — Pull your repo

Set `GITHUB_USER` below. The repo must have `src/train_v27_colab.py` and all wrapper files.

**Private repo?** Create a GitHub [fine-grained token](https://github.com/settings/tokens?type=beta) with Contents:Read on the repo, and swap the URL to `https://<token>@github.com/<user>/Rocket.git`.

In [ ]:
GITHUB_USER = '[YOUR-GITHUB-USERNAME]'    # <-- EDIT ME
REPO_NAME   = 'Rocket'                     # <-- EDIT if your repo is named differently
BRANCH      = 'main'

!rm -rf /content/rocket
!git clone --branch {BRANCH} --depth 1 https://github.com/{GITHUB_USER}/{REPO_NAME}.git /content/rocket

%cd /content/rocket/src
!ls train_v27_colab.py target_landing.py sensor_noise.py actuator_noise.py safety_shield.py thrust_scaled_lander.py

## Cell 4 — Launch TensorBoard (background, before training)

Opens inline. Watch `ep_rew_mean`, `eval/mean_reward`, and curriculum transitions.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/rocket/logs/tb_v27

## Cell 5 — Wiring test (200k steps, ~10 minutes)

**Run this first, always.** Confirms SubprocVecEnv spawns, VecFrameStack produces 72-dim obs, curriculum callback logs, checkpoints write to Drive, EvalCallback runs.

In [ ]:
import os
os.environ['V27_OUT']   = '/content/drive/MyDrive/rocket'
os.environ['V27_STEPS'] = '200000'
os.environ['V27_NENVS'] = '4'
os.environ['V27_SEED']  = '7474'

!python train_v27_colab.py

## Cell 6 — Full 8M training run (~10–12 h on T4)

The real run. Watch TensorBoard for the first 15 minutes to confirm reward is climbing, then let it run. Checkpoints to Drive every 100k steps — a disconnect loses at most a few minutes.

In [ ]:
import os
os.environ['V27_OUT']   = '/content/drive/MyDrive/rocket'
os.environ['V27_STEPS'] = '8000000'
os.environ['V27_NENVS'] = '4'
os.environ['V27_SEED']  = '7474'

# If SubprocVecEnv throws on Colab, uncomment to fall back to DummyVecEnv (~60% speed):
# os.environ['V27_SUBPROC'] = '0'

!python train_v27_colab.py

## Cell 7 — Keep-alive (optional)

Clicks a hidden Connect button once per minute so Colab doesn't idle out mid-training. Run in a separate browser tab.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    console.log("Keeping Colab alive");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
'''))

## Cell 8 — Sanity-check the best model in Colab

Loads `master_v27_best.zip` and runs 100 eval episodes at full noise, target_limit=0.10. Expect ~98–99% precise.

In [ ]:
import os, sys
sys.path.insert(0, '/content/rocket/src')
os.environ['V27_OUT'] = '/content/drive/MyDrive/rocket'

import numpy as np
from stable_baselines3 import SAC
from train_v27_colab import build_eval_vec, BEST_MODEL

model = SAC.load(str(BEST_MODEL), device='cuda')
env   = build_eval_vec(seed=99999)

N = 100
precise = safe_out = crash = timeout = leg = 0
rewards = []

for ep in range(N):
    obs = env.reset()
    ep_r = 0.0
    done = False
    info = {}
    while not done:
        act, _ = model.predict(obs, deterministic=True)
        obs, r, dones, infos = env.step(act)
        ep_r += float(r[0])
        done = bool(dones[0])
        info = infos[0]
    rewards.append(ep_r)
    outcome = info.get('terminal_outcome') or info.get('outcome') or 'OTHER'
    if   outcome == 'PRECISE':      precise += 1
    elif outcome == 'SAFE_OUTSIDE': safe_out += 1
    elif outcome == 'CRASH':        crash   += 1
    elif outcome == 'TIMEOUT':      timeout += 1
    elif outcome == 'LEG_FAILURE': leg += 1

print(f'\n=== V27 BEST @ full noise, target_limit=0.10 (n={N}) ===')
print(f'  PRECISE       {precise/N:.1%}')
print(f'  SAFE_OUTSIDE  {safe_out/N:.1%}')
print(f'  LEG_FAILURE   {leg/N:.1%}')
print(f'  CRASH         {crash/N:.1%}')
print(f'  TIMEOUT       {timeout/N:.1%}')
print(f'  mean reward   {np.mean(rewards):.1f}')

## Cell 9 — Download the best model to your laptop

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/rocket/models/master_v27_best.zip')
files.download('/content/drive/MyDrive/rocket/models/master_v27_best_manifest.json')

## After the run — on your Windows laptop

Copy the best model and evaluate at the ship bar (target_limit=0.15):

```powershell
copy "<downloaded_folder>\master_v27_best.zip" models\master_v27_best.zip

.venv\Scripts\python.exe src\evaluate_all.py --model models\master_v27_best.zip --profile v23 --episodes 1000 --seed-offset 20000 --shield    --out logs\v27_best_common.csv
.venv\Scripts\python.exe src\evaluate_all.py --model models\master_v27_best.zip --profile v23 --episodes 1000 --seed-offset 20000 --no-shield --out logs\v27_best_noshield.csv
```

Use `--profile v23` because v27's env config matches v23 exactly — only the policy architecture and training scale differ.

---

**Success bar:** ≥98% precise (shield-off) on 1000 episodes. If v27 clears this, it is your Phase-1 production model and v23 is retired.